<img src="logo.png" alt="Vegeta" width="240">

# Mini raptors — Apus Mini, Alcotan, Gavilan: three first-guess designs on a 5-inch racing stack

Three small tailsitters that reuse what a 5-inch FPV racer already owns — four 2207-class motors, 5-inch tri-blade
propellers, the 4-in-1 ESC and flight controller, the FPV camera and video link, a 4S or 6S pack — and spend the frame
on a wing. Each has its own brief and its own layout; none is forced into the other's:

| design | brief | layout chosen |
|---|---|---|
| **Apus Mini** | lower energy than the stock racer at comparable speed and manoeuvre | biplane tailsitter, the widest cell a racing gate allows, 4S 1500 |
| **Alcotan** | faster sustained flight for pursuit | cruciform (+) tailsitter: a small, highly loaded wing and two fins carry the motors at the arm tips; 6S, the 5x5 |
| **Gavilan** | maximum practical agility in hangars and industrial courses | the racing frame's motor square wrapped in a short biplane cell, the lightest pack; the wing is a brake and a glider |

The reference is the racing quad itself, modelled on the same parts (§ 2); the published table — 300–500 g,
35–55 m/s, 41 m at 45° bank and 20 m/s, 2–5 km at economical cruise — is a check on that model, not an input.

**Method and its limits.** First principles and empirical correlations only (`designs/mini_raptor.py`): flat-plate
skin friction with form factors at the chord Reynolds number, Prandtl's biplane factor, Boreas's blade-element tables
for the drives (made once, cached), rigid-body hover accelerations, point-mass braking. **No CFD, no FEA, no
optimisation, no long simulation.** The CAD is conceptual — envelopes and surfaces, not a validated structure. Earlier
claims (420 g, 25 W, 40–55 km) are hypotheses this notebook tests, and the uncertainty table (§ 6) says how far to
trust each number. Everything runs in about a minute.

In [ ]:
import os, sys, time, json, math
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
sys.path.insert(0, "designs")
import mini_raptor as mr
import kestrel_report as kr
from vegeta import yanus
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 80); pd.set_option("display.max_colwidth", 70)
T0 = time.time()
GATE = yanus.enter("49_mini_raptor")            # nothing to take: the notebook computes everything it shows
display(GATE.table())

## 1. The stack and the drives

The parts every design reuses, and what the four drive options give: static thrust, thrust at speed (it falls towards
the propeller's pitch speed), the electrical power at the hover's low thrust (where racing motors are at their least
efficient). The tables come from Boreas once and are cached in `designs/data/mini_raptor_drive_*.json`.

In [ ]:
display(pd.DataFrame(mr.STACK).T[["name", "mass_kg"]])
display(pd.DataFrame(mr.PACKS).T)
DRV = pd.DataFrame({k: mr.drive_summary(k) for k in mr.DRIVES})
display(DRV.round(2))
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for k in mr.DRIVES:
    t = mr.drive_table(k)
    ax[0].plot(t.V, t.T_max, label=k)
    ax[1].plot(t.T, [t.power(x, 20.0) for x in t.T], label=f"{k} at 20 m/s")
ax[0].set(xlabel="axial inflow [m/s]", ylabel="full-throttle thrust per motor [N]", title="thrust against speed"); ax[0].grid(alpha=0.3); ax[0].legend()
ax[1].set(xlabel="thrust per motor [N]", ylabel="electrical [W]", title="power against thrust at 20 m/s", xlim=(0, 6), ylim=(0, 250)); ax[1].grid(alpha=0.3); ax[1].legend()
plt.show()
print(f"§ 1 in {time.time() - T0:.0f} s")

## 2. The racing quad, the fair baseline

The same stack on a 5-inch carbon frame (its mass budget below), flown tilted: the body's drag area (0.016 m² at
racing tilt, ± 30 %) sets the tilt, the tilt sets the thrust, the drives' table gives the power at the axial inflow.
The pilot's tilt limit (70°) caps the top speed and the brake; the turn is thrust-to-weight, not a wing. Against the
published table: the model's top speed and range should fall in or near its bands — if they do not, the baseline, not
the minis, is what to doubt first.

In [ ]:
BASE = mr.quad()
display(pd.DataFrame(mr.quad_mass_table())[["item", "mass_kg", "group", "note"]].assign(mass_g=lambda d: (d.mass_kg * 1e3).round(1)).drop(columns="mass_kg"))
display(pd.Series(BASE["summary"], name="racing quad").to_frame().T.T.round(3))
display(BASE["table"].loc[[5, 10, 15, 20, 25, 30, 35]].round(2))
SENS_Q = pd.DataFrame({f"CdA {c:.3f} m²": {k: mr.quad(CdA_m2=c)["summary"][k] for k in ("power at 20 m/s [W]", "Wh/km at 20 m/s", "best range [km]", "top speed [km/h]", "sustained top speed [km/h]")} for c in (0.011, 0.016, 0.021)})
display(SENS_Q.round(2))
CHECK = pd.DataFrame({"published band": {"mass [g]": "300–500", "top speed [m/s]": "35–55", "turn radius 45° bank, 20 m/s [m]": "41", "economical range [km]": "2–5"},
                      "this model": {"mass [g]": f"{BASE['summary']['mass [kg]'] * 1e3:.0f}", "top speed [m/s]": f"{BASE['summary']['top speed [m/s]']:.0f} (tilt-limited; sustained {BASE['summary']['sustained top speed [m/s]']:.0f})",
                                     "turn radius 45° bank, 20 m/s [m]": f"{BASE['summary']['turn radius at 20 m/s, 45° bank [m]']:.0f}", "economical range [km]": f"{BASE['summary']['best range [km]']:.1f} at {BASE['summary']['at [m/s]']:.0f} m/s"}})
display(CHECK)
print(f"§ 2 in {time.time() - T0:.0f} s")

The model's economical range sits above the published 2–5 km: a racer at economical cruise is still a racer
(the body's drag area, the part-throttle efficiency of 2207s, the pilot's throttle); the published band is flown, this
is computed. The gap is the uncertainty to carry into the comparison — about a factor of two on range for the quad,
less for the winged designs whose cruise is a wing's.

## 3. The three designs

For each: the layout and its rules (the propeller discs: two 5-inch discs stacked need 142 mm, four side by side on two
wings need 360 mm of span — that rule shaped all three), the component mass budget, the drag build-up, then the
conceptual CAD: a labelled 3-D render, a cutaway with the component envelopes, dimensioned orthographic views, STEP
and STL in `designs/data/cad/mini_raptor/`.

In [ ]:
PERF = {v: mr.performance(v) for v in mr.VARIANTS}
display(mr.design_rows(PERF).T)
CAD = {}
def show(v):
    p = PERF[v]; L = p["L"]
    display(Markdown(f"### {v} — {L['title']}"))
    b = mr.budget(L)
    display(b.groupby("group")["mass_g"].sum().to_frame().T.assign(total=b["mass_g"].sum()).round(1))
    display(b)
    d = mr.drag_buildup(L)
    display(pd.Series({k: x * 1e4 for k, x in d["parts_m2"].items()} | {"total with interference": d["CdA_m2"] * 1e4}, name="drag area at 20 m/s [cm²]").to_frame().T.round(2))
    display(pd.Series({k: p["summary"][k] for k in ("mass [kg]", "wing loading [N/m²]", "aspect ratio", "Cd0 at 20 m/s", "induced k", "stall speed [m/s]", "hover thrust / weight", "hover power [W]")}, name=v).to_frame().T.round(3))
    prt = mr.parts(mr.cad_params(v))
    fig = plt.figure(figsize=(16, 7))
    mr.render3d(prt, fig.add_subplot(1, 2, 1, projection="3d"), title=f"{v}: labelled render")
    mr.render3d(prt, fig.add_subplot(1, 2, 2, projection="3d"), title=f"{v}: cutaway — the component envelopes", cutaway=True, labels=False, elev=18, azim=-35)
    plt.show()
    mr.ortho_views(prt, L, title=f"{v}: dimensioned views [mm]"); plt.show()
    CAD[v] = mr.export_cad(v)
    print("CAD:", CAD[v])
show("Apus Mini")
print(f"§ 3 Apus in {time.time() - T0:.0f} s")

In [ ]:
show("Alcotan")
print(f"§ 3 Alcotan in {time.time() - T0:.0f} s")

In [ ]:
show("Gavilan")
print(f"§ 3 Gavilan in {time.time() - T0:.0f} s")

## 4. Against speed: power, Wh/km, endurance, range, load factor, turn radius

The three winged designs and the quad on the same axes. The turn here is the **banked turn** at the load factor
available (the lesser of the wing's CL max, the thrust-sustained turn and the structural limit); the hover
reorientation is § 5.

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(18, 9))
for v, p in PERF.items():
    t = p["table"][p["table"]["feasible"]]; col = p["L"]["colour"]
    ax[0, 0].plot(t.index, t["electrical [W]"], color=col, label=v); ax[0, 1].plot(t.index, t["Wh/km"], color=col, label=v); ax[0, 2].plot(t.index, t["range [km]"], color=col, label=v)
    ax[1, 0].plot(t.index, t["L/D"], color=col, label=v); ax[1, 1].plot(t.index, t["n"], color=col, label=v); ax[1, 2].plot(t.index, t["turn radius [m]"], color=col, label=v)
    ax[0, 0].axhline(p["summary"]["sustained power [W]"], color=col, ls=":", lw=0.8)
q = BASE["table"]
ax[0, 0].plot(q.index, q["electrical [W]"], "k--", label="racing quad"); ax[0, 1].plot(q.index, q["Wh/km"], "k--", label="racing quad"); ax[0, 2].plot(q.index, q["range [km]"], "k--", label="racing quad")
ax[1, 2].plot(q.index, [BASE["summary"]["turn radius at 20 m/s (banked) [m]"] * (V / 20) ** 2 for V in q.index], "k--", label="racing quad (thrust-limited tilt)")
for a, (yl, ylim) in zip(ax.flat, (("electrical [W] (dotted: sustainable)", (0, 1000)), ("Wh per km", (0, 4)), ("range [km]", (0, 60)), ("L/D", (0, 12)), ("load factor available", (0, 12)), ("banked turn radius [m]", (0, 80)))):
    a.set(xlabel="airspeed [m/s]", ylabel=yl, ylim=ylim, xlim=(0, 85)); a.grid(alpha=0.3); a.legend(fontsize=8)
plt.tight_layout(); plt.show()
for v, p in PERF.items():
    t = p["table"]; rows = [V for V in (12, 15, 20, 25, 30, 40, 50, 60, 70) if V in t.index]
    display(Markdown(f"**{v}**")); display(t.loc[rows, ["feasible", "CL", "L/D", "drag [N]", "thrust margin", "electrical [W]", "Wh/km", "endurance [min]", "range [km]", "n", "turn radius [m]", "turn rate [deg/s]", "limited by", "note"]].round(2))
print(f"§ 4 in {time.time() - T0:.0f} s")

## 5. Manoeuvre: banked turns against hover reorientation, braking, transitions

Two different things a tailsitter and a quad both do. A **banked turn** at speed is a wing's business: radius
V² / (g √(n² − 1)) at the load factor available. A **hover reorientation** is a rotation on the spot — the nose pointed
elsewhere with differential thrust — and takes a fraction of a second for all four aircraft; the quad also yaws
(torque only, slower). The **brake** from 20 m/s: the tailsitter pitches to the vertical and beyond (the thrust tilted
back like the quad's, up to 70°) with the whole wing as an air brake; the quad tilts back and has only its body's drag.
The **transitions**: thrust to weight at 1.2 × the stall speed out of the hover, and the pull-up radius into it.

In [ ]:
MAN = mr.compare(PERF, BASE).loc[["n at 20 m/s (banked)", "turn radius at 20 m/s (banked) [m]", "turn radius at 40 m/s (banked) [m]", "least banked turn radius [m]",
                                    "pitch acceleration in the hover [rad/s²]", "hover pitch-over 90° [s]", "brake deceleration at 20 m/s [g]", "brake 20→4 m/s distance [m]", "brake 20→4 m/s time [s]",
                                    "stall speed [m/s]", "thrust / weight at 1.2 Vs", "pull-up speed [m/s]", "pull-up radius into the hover [m]", "transitions"]]
MAN.loc["hover yaw 90° [s] (quad: torque)"] = [BASE["summary"].get("hover yaw 90° [s]"), None, None, None]
MAN.loc["hover roll 90° [s]"] = [None] + [p["summary"]["hover roll 90° [s]"] for p in PERF.values()]
display(MAN)
print(f"§ 5 in {time.time() - T0:.0f} s")

## 6. Comparison, targets, uncertainty

In [ ]:
CMP = mr.compare(PERF, BASE)
display(CMP)
TGT = mr.targets(PERF, BASE)
display(TGT)
display(pd.Series(mr.UNCERTAINTY, name="relative uncertainty assumed").to_frame().T)
SENS = {v: mr.sensitivity(v) for v in mr.VARIANTS}
for v, s in SENS.items():
    display(Markdown(f"**{v}: sensitivity**")); display(s.round(2))
print(f"§ 6 in {time.time() - T0:.0f} s")

## 7. Findings

Read against the tables above (recomputed on every run):

- **A wing turns the racer's energy problem around.** At 20 m/s the quad spends over 2 Wh per km holding itself up
  with tilted thrust; the three winged designs spend 0.5–0.7. Apus Mini's brief is met with room to spare — the
  economy comes from the wing, not from any refinement — and its banked turn at 20 m/s is tighter than the quad's
  45°-bank figure. The price is span: a third of a gate.
- **Alcotan's sustained speed is the 5x5's pitch speed and the pack's C rating, not the airframe.** The cruciform
  carries the four motors on four tips with nothing between the discs, and at the sustainable power (a 30C pack, 22 A
  per motor) it runs a good deal faster than the quad can sustain; the thrust-limited burst speed above that empties
  the pack in under a minute and is reported as a burst. Its stall is high and its slow turn poor: it is a chaser, not
  a racer in a hangar, and the pull-up into the hover from its cruise is wide.
- **Gavilan cannot match the quad's hover agility and the model says so.** The racing frame's motor square gives it
  the quad's thrust arms, but the cell and the pod add inertia: about half the quad's pitch acceleration — still a
  90° pitch-over in a fraction of a second, well inside a pilot's reaction. What the wing buys in a hangar is the
  brake (the wing at 90° with the thrust tilted back stops it shorter than the quad from 20 m/s) and a slow,
  economical glide between obstacles; what it costs is 60 mm more span than the quad's frame and a stall speed the
  quad does not have. "Maximum practical agility" in tight spaces is, by this model, mostly the quad's thrust-to-weight,
  which all three keep.
- **None of the three is 420 g.** The stack alone is 300 g with the pack; the wings, posts, pod and feet add 150–170 g
  at the assumed densities (± 20 %). 500–570 g is the honest band; the quad on the same parts is 500 g.
- **What this model cannot tell:** the thin sections' cl max at Reynolds numbers near 100 000 (± 10 % assumed, and it
  moves the stall and the slow turn directly), the drag build-up at ± 30 % (it moves the economy and the top speed),
  the 5-inch drives from a blade section calibrated at 9 inches, and every structural number. Those are the next
  notebook's questions, with a section polar (Zephyr) and a bare-airframe CFD, if any of the three goes further.

In [ ]:
OUT = mr.export_design(PERF, BASE, CAD)
rep = kr.export("49_mini_raptor", "Mini raptors — Apus Mini, Alcotan, Gavilan: three first-guess designs on a 5-inch racing stack",
                {"drives": DRV, "racing quad": pd.Series(BASE["summary"]).to_frame("racing quad"), "designs": mr.design_rows(PERF), "comparison": CMP, "targets": TGT, "manoeuvre": MAN,
                 "sensitivity": {v: s for v, s in SENS.items()}, "uncertainty": pd.Series(mr.UNCERTAINTY).to_frame("relative")},
                meta={"method": "first principles and empirical correlations; no CFD, no FEA; conceptual CAD", "cad": CAD})
print("design file:", OUT); print("report:", rep)
GATE.leave()
print(f"done in {time.time() - T0:.0f} s")